# RQ1 — presence-vs-background classifier (citizen science)

**Purpose.** RQ1's redesign (2026-11-05/06) runs a true one-class
classifier **and** presence-vs-background in parallel, not staged, for
every point of the effort-reduction curve (see
[`thesis/notes.md`](../../thesis/notes.md#multiclasse-abandonado-rq1-passa-a-uma-curva-de-redução-de-esforço-presença-vs-fundo)).
The one-class side (one-class SVM, chosen over Isolation Forest) is
already tested for the citizen-science arm in
[`one_class_method_selection.ipynb`](one_class_method_selection.ipynb).
This notebook adds the missing half for the same arm: a
presence-vs-background classifier, reusing the exact same background
pool and spatial-block splits, so the two methods are evaluated the
same way and are directly comparable.

**Scope of this first pass:** citizen-science *A. dealbata* presences
only — same scope as the one-class notebook. The expert arm and the
full effort-reduction curve are separate, later work (see
`docs/todo.md`).

**Candidate classifiers (`docs/research_questions.md#rq1--foundation-models-vs-conventional-approaches`):**
Random Forest and Logistic Regression are compared here, both already
listed as candidates in `docs/research_questions.md`. Random Forest is
grounded there in Ball et al. (2026), Yang et al. (2026), Mouta et al.
(2021), Domingo et al. (2023) and Eskandari et al. (2025). **Logistic
Regression has no literature grounding checked yet** — it is listed as
a candidate but, unlike Random Forest and the compact neural network,
`docs/research_questions.md` names no supporting work for it; it is
included here as a simple, standard linear baseline, not because a
specific paper was found recommending it for this setting. **The third
candidate, a compact neural network, is deferred to later work** — not
tested here, to keep this first pass proportionate to its purpose
(confirm the arm works, pick a classifier to carry into the curve),
the same scoping choice as the one-class notebook's own hyperparameter
sweep.


## 1. Background pool, presences, and TESSERA embeddings

Reuses, without rebuilding, the exact files `one_class_method_selection.ipynb`
already produced:
- the 2 000-point shared background pool (uniform random, 50 m buffer
  around every known presence — see `data/README.md`, not yet
  confirmed by the supervisor as the final design);
- the RQ1 pilot's TESSERA embeddings (citizen-science, expert and old
  COS-negative points).

If the background file is missing, run `one_class_method_selection.ipynb`
first — this notebook does not rebuild it, to avoid duplicating that
logic in two places.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.embeddings import block_id, load_pilot_embeddings, match_to_pixel

EMB_DIR = ROOT / "data" / "processed" / "embeddings"
PILOT_DIR = EMB_DIR / "tessera_v1.1_2023_rq1_pilot"
BG_FILE = EMB_DIR / "tessera_v1.1_2023_rq1_background" / "background_points.npz"

if not BG_FILE.exists():
    raise FileNotFoundError(
        f"{BG_FILE} is missing — run one_class_method_selection.ipynb first, "
        "it builds the shared background pool this notebook reuses."
    )

pilot_df, pilot_emb = load_pilot_embeddings(PILOT_DIR)
bg = np.load(BG_FILE)
bg_x, bg_y, bg_emb = bg["x"], bg["y"], bg["emb"]

dealbata = pd.read_csv(ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv")
pres_x, pres_y, pres_emb, n_matched = match_to_pixel(dealbata["pixel"], pilot_df, pilot_emb)

print(f"presences matched to an embedding: {n_matched}/{len(dealbata)}")
print(f"presences with a non-NaN embedding: {len(pres_emb)}")
print(f"background points with valid embeddings: {len(bg_emb):,}")


## 2. Spatial block holdout

Same 10 km UTM blocks and the same 5 block-holdout seeds (20% of
blocks held out as the test set) as `one_class_method_selection.ipynb`,
so the two methods' results are comparable, not an artefact of
different splits.


In [ ]:
pres_block = block_id(pres_x, pres_y)
bg_block = block_id(bg_x, bg_y)
all_blocks = sorted(set(pres_block) | set(bg_block))
print(f"{len(all_blocks)} spatial blocks (10 km) covering presences + background")


## 3. Compare Random Forest vs. Logistic Regression

Both trained on presence (label 1) **and** background (label 0)
embeddings from the training blocks — unlike the one-class arm, this
is a genuine two-class classifier. `class_weight="balanced"` for both,
since presences (2 995) and background (2 000) are not far from
balanced but not equal either. Evaluated on the held-out test-block
presences vs. the held-out test-block background, with the same two
metrics as the one-class notebook:

- **ROC-AUC**.
- **Continuous Boyce index** (Hirzel et al., 2006), using the
  predicted probability of presence as the score.

Random Forest does not need feature scaling; Logistic Regression does
— both are given the same standardised features (fit on the training
fold) for a fair comparison, scaling being a no-op for the forest.
Hyperparameters are not tuned here beyond sensible defaults — exact
tuning is left to later work once the full pipeline (curve, expert
arm) exists, matching the one-class notebook's own scoping.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

from src.evaluation import continuous_boyce

configs = [
    ("RandomForest", lambda: RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)),
    ("LogisticRegression", lambda: LogisticRegression(class_weight="balanced", max_iter=1000)),
]

results = {name: {"auc": [], "boyce": []} for name, _ in configs}
for seed in range(1, 6):
    rng = np.random.default_rng(seed)
    test_blocks = set(rng.choice(all_blocks, size=max(1, round(0.2 * len(all_blocks))), replace=False))
    train_mask_pres, test_mask_pres = ~np.isin(pres_block, list(test_blocks)), np.isin(pres_block, list(test_blocks))
    train_mask_bg, test_mask_bg = ~np.isin(bg_block, list(test_blocks)), np.isin(bg_block, list(test_blocks))

    pres_train, pres_test = pres_emb[train_mask_pres], pres_emb[test_mask_pres]
    bg_train, bg_test = bg_emb[train_mask_bg], bg_emb[test_mask_bg]
    if len(pres_test) < 5 or len(bg_test) < 5:
        continue

    X_train = np.concatenate([pres_train, bg_train])
    y_train = np.concatenate([np.ones(len(pres_train)), np.zeros(len(bg_train))])
    X_test = np.concatenate([pres_test, bg_test])
    y_test = np.concatenate([np.ones(len(pres_test)), np.zeros(len(bg_test))])

    scaler = StandardScaler().fit(X_train)
    X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

    for name, ctor in configs:
        model = ctor().fit(X_train_s, y_train)
        score = model.predict_proba(X_test_s)[:, 1]
        score_pres, score_bg = score[y_test == 1], score[y_test == 0]
        results[name]["auc"].append(roc_auc_score(y_test, score))
        results[name]["boyce"].append(continuous_boyce(score_pres, score_bg))

summary = pd.DataFrame(
    {
        "ROC-AUC (mean)": [np.mean(results[n]["auc"]) for n, _ in configs],
        "ROC-AUC (sd)": [np.std(results[n]["auc"]) for n, _ in configs],
        "Boyce (mean)": [np.mean(results[n]["boyce"]) for n, _ in configs],
        "Boyce (sd)": [np.std(results[n]["boyce"]) for n, _ in configs],
        "n_splits": [len(results[n]["auc"]) for n, _ in configs],
    },
    index=[n for n, _ in configs],
)
summary.round(3)


## 4. Decision

**Result.** Both classifiers score far higher than the one-class arm
tested in `one_class_method_selection.ipynb` (ROC-AUC ≈0.64–0.65,
Boyce ≈0.6–0.8): Random Forest ROC-AUC 0.842 (sd 0.028), Boyce 0.989
(sd 0.022); Logistic Regression ROC-AUC 0.878 (sd 0.052), Boyce 0.973
(sd 0.042) — averaged over the same 5 spatial-block splits. This
matches what the literature search already found for true one-class
methods without background (see `thesis/notes.md`'s "Atualização",
2026-11-06): having background data to train against clearly helps
here too, for this species and representation.

**Random Forest and Logistic Regression are close, not a clear win for
either:** Logistic Regression has the higher mean ROC-AUC but also
clearly higher variance across splits; Random Forest has the higher
mean Boyce index and is more stable (lower sd on both metrics).
Neither dominates on both axes.

**Proposal (my own, first pass, not confirmed by the supervisor):**
carry **Random Forest** forward as the default presence-vs-background
classifier for the rest of RQ1 (expert arm, full curve), because (1)
it is more stable across spatial splits, which matters more than a
small mean-AUC gap given how few independent spatial blocks the
expert side has (15, see `expert_polygons.ipynb`), and (2) it already
has five supporting citations in `docs/research_questions.md`, whereas
Logistic Regression has none checked. **Logistic Regression is not
discarded** — it stays a reported baseline (cheap to keep computing
alongside Random Forest), since the gap is small enough that this
could change once the expert arm and the full curve are tested.

**Caveats (first pass, not a definitive benchmark):**

- Citizen-science *A. dealbata* only — the expert arm and the full
  effort-reduction curve are not tested here.
- The background pool (uniform random, 50 m buffer) is the same one
  used in `one_class_method_selection.ipynb`, still not confirmed by
  the supervisor as the final shared design.
- Hyperparameters are not tuned (`n_estimators=300` for the forest,
  default `C=1.0` for Logistic Regression) — just sensible defaults,
  no sweep.
- The compact neural network candidate from `docs/research_questions.md`
  is not tested here (see "Purpose" above).
